In [1]:
# =============================================================================
# CELL 2 — INSTALL DEPENDENCIES
# =============================================================================

!pip install -q uv

!uv pip install --system \
    "transformers<4.40.0" \
    "peft<=0.10.0" \
    "accelerate>=0.28.0,<0.30.0" \
    datasets \
    sacrebleu \
    sentencepiece \
    einops \
    pandas \
    numpy \
    scikit-learn \
    tqdm \
    sentence-transformers

!uv pip install --system \
    "git+https://github.com/VarunGumma/IndicTransToolkit.git"

print("=" * 80)
print("DEPENDENCIES INSTALLED")
print("=" * 80)

import torch
import transformers
import peft
import sacrebleu

print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("SacreBLEU    :", sacrebleu.__version__)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print(
        "BF16 support :",
        torch.cuda.is_bf16_supported()
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 89.8 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 83 packages in 666ms
Prepared 8 packages in 230ms
Uninstalled 6 packages in 890ms
Installed 8 packages in 45ms
 - accelerate==1.14.0
 + accelerate==0.29.3
 - huggingface-hub==1.29.0
 + huggingface-hub==0.36.2
 - peft==0.20.0
 + peft==0.10.0
 + portalocker==4.4.0
 + sacrebleu==2.6.0
 - sentence-transformers==5.7.0
 + sentence-transformers==3.1.1
 - tokenizers==0.23.1
 + tokenizers==0.15.2
 - transformers==5.16.1
 + transformers==4.39.3
Using Python 3.13.15 environment at: /usr
Resolved 56 packages in 531ms
Prepared 7 packages in 9.91s
Installed 7 packages in 5ms
 + indic-nlp-library-itt==0.1.1
 + indictranstoolkit==1.1.1 (from git+https://github.com/VarunGumma/IndicTransToolkit.git@3efb8418d0721b4ce267c2b3586899d313191357)
 + morfessor==2.0.6
 + sacremoses==0.2.0
 + sphinx-argparse==0.6.1
 + sphinx-rtd-theme==3.1.0
 + sphinxcontrib-jquery==4.1
DEPENDENCIES IN

In [2]:
# =============================================================================
# CELL 3 — GLOBAL CONFIGURATION   (UPDATED)
#
# Changes vs. the original:
#   * All training / filtering hyper-parameters are now centralised here.
#   * DIRECTION_OUTPUT_ROOT points to a NEW folder (..._v2) so that the old
#     low-LR checkpoints are never auto-resumed by mistake.
#   * LoRA now uses a realistic LoRA learning rate (2e-4, not 2e-5).
# =============================================================================

import os
import gc
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# DEVICE
# =============================================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# MODEL DTYPE  (T4 has no bf16 -> fp16 is used automatically)
# =============================================================================

# NOTE: torch.cuda.is_bf16_supported() returns True on a T4 (software
# emulation) but T4 has NO bf16 hardware -> bf16 there is extremely slow.
# Use bf16 only on Ampere or newer (compute capability >= 8).
_cc_major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0

if torch.cuda.is_available() and _cc_major >= 8 and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
else:
    MODEL_DTYPE = torch.float16

print("Model dtype:", MODEL_DTYPE)


# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

QUICK_TEST_MODE = False          # True -> tiny smoke test of the whole pipeline

# Distilled 320M Indic-Indic model (~3x faster than the 1B model).
# NOTE: the repo is GATED -> open the model page once, accept the conditions
# with the same Hugging Face account that owns your HF_TOKEN Kaggle secret.
MODEL_NAME = "ai4bharat/indictrans2-indic-indic-dist-320M"


# =============================================================================
# DATA-QUALITY / SPLIT SETTINGS
# =============================================================================

# LaBSE filter: a pair is dropped if its similarity is below
#   max(SIM_FLOOR, the DROP_BOTTOM_PCT-th percentile of all similarities)
SIM_FLOOR = 0.55
DROP_BOTTOM_PCT = 5

# Use as much data as possible for training (the base model is strong, so the
# extra in-domain sentences matter more than a huge validation set).
VAL_FRAC = 0.05
TEST_FRAC = 0.05


# =============================================================================
# LoRA / TRAINING SETTINGS
# =============================================================================

LORA_R = 96                      # smaller model -> give the adapter more capacity
LORA_ALPHA = 128
LORA_DROPOUT = 0.1

LEARNING_RATE = 2e-4             # LoRA needs ~1e-4 .. 5e-4  (2e-5 = no learning)
NUM_EPOCHS = 5
PER_DEVICE_BATCH = 8
GRAD_ACCUM = 2                   # effective batch = 32
MAX_TRAIN_PER_DIRECTION = None   # e.g. 12000 to cap training size for speed
WARMUP_RATIO = 0.05
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 2          # stop if val chrF++ fails to improve 2 epochs
VAL_BEAMS = 4                    # beams used only for per-epoch selection


# =============================================================================
# LANGUAGE TAGS
# =============================================================================

LANG_TAG = {
    "guj": "guj_Gujr",
    "mar": "mar_Deva",
    "doi": "doi_Deva",
    "pan": "pan_Guru",
}


# =============================================================================
# KAGGLE OUTPUT DIRECTORIES
# =============================================================================

OUTPUT_DIR = Path("/kaggle/working/experiment1_indictrans2_lora")

SPLIT_DIR = OUTPUT_DIR / "splits"

# NEW folder -> checkpoints of the earlier 1B runs are NOT resumed
DIRECTION_OUTPUT_ROOT = OUTPUT_DIR / "direction_specific_lora_320m_v2_translit"

RESULTS_DIR = OUTPUT_DIR / "results"

for directory in [OUTPUT_DIR, SPLIT_DIR, DIRECTION_OUTPUT_ROOT, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# DATASET
# =============================================================================

DATASET_PATH = Path("/kaggle/input/datasets/rkstriker/setunlp")

FILE_MAP = {
    "doi": DATASET_PATH / "train-dogri.txt",
    "pan": DATASET_PATH / "train-punjabi.txt",
    "guj": DATASET_PATH / "train-gujarathi.txt",
    "mar": DATASET_PATH / "train-marathi.txt",
}


# =============================================================================
# VALIDATE DATASET FILES
# =============================================================================

for language, filepath in FILE_MAP.items():

    if not filepath.exists():
        raise FileNotFoundError(
            f"Missing required data file: {filepath.resolve()}"
        )

    print(f"{language:>4s} -> {filepath}")


print("=" * 80)
print("GLOBAL CONFIGURATION READY")
print("=" * 80)


Device: cuda
GPU: Tesla T4
Model dtype: torch.float16
 doi -> /kaggle/input/datasets/rkstriker/setunlp/train-dogri.txt
 pan -> /kaggle/input/datasets/rkstriker/setunlp/train-punjabi.txt
 guj -> /kaggle/input/datasets/rkstriker/setunlp/train-gujarathi.txt
 mar -> /kaggle/input/datasets/rkstriker/setunlp/train-marathi.txt
GLOBAL CONFIGURATION READY


In [3]:
# -----------------------------------------------------------------------------
# CELL 4 — Data Ingestion, Normalisation & Quality Filtering   (UPDATED)
#
# Changes vs. the original:
#   * NFC instead of NFKC (NFKC splits nukta letters such as क़ / ਖ਼ into
#     base+nukta, which silently changes the words you are scored against).
#   * Line-alignment check between the two files of every pair.
#   * Extra parallel-data filters: empty lines, copy (src == tgt),
#     very short / very long lines, extreme length ratio.
#   * Statistics are printed so you can SEE how much noise was removed.
# -----------------------------------------------------------------------------
import re
import unicodedata


def load_raw(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]


raw = {lang: load_raw(path) for lang, path in FILE_MAP.items()}

for lang, lines in raw.items():
    print(f"{lang}: {len(lines):,} raw lines")

assert len(raw["guj"]) == len(raw["mar"]), "guj / mar files are not line-aligned!"
assert len(raw["doi"]) == len(raw["pan"]), "doi / pan files are not line-aligned!"


# -----------------------------------------------------------------------------
# Text cleaning
# -----------------------------------------------------------------------------

PURE_NOISE_RE = re.compile(r"[\u200b\u2060\ufeff]")
WATERMARK_EDGES_RE = re.compile(r"^[\u200c\u200d]+|[\u200c\u200d]+$")


def clean_text(s: str) -> str:
    s = PURE_NOISE_RE.sub("", s)
    s = WATERMARK_EDGES_RE.sub("", s)
    s = unicodedata.normalize("NFC", s)          # NFC, NOT NFKC
    return re.sub(r"\s+", " ", s).strip()


clean = {lang: [clean_text(l) for l in lines] for lang, lines in raw.items()}


# -----------------------------------------------------------------------------
# Script validation
# -----------------------------------------------------------------------------

DEVA_RANGE = (0x0900, 0x097F)
GURM_RANGE = (0x0A00, 0x0A7F)
GUJR_RANGE = (0x0A80, 0x0AFF)


def is_valid_line(text: str, lang: str) -> bool:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return False
    total = len(letters)

    if lang == "pan":
        gurm = sum(1 for c in letters if GURM_RANGE[0] <= ord(c) <= GURM_RANGE[1])
        deva = sum(1 for c in letters if DEVA_RANGE[0] <= ord(c) <= DEVA_RANGE[1])
        if deva / total > 0.15:
            return False
        return (gurm / total) >= 0.25

    if lang in ["doi", "mar"]:
        deva = sum(1 for c in letters if DEVA_RANGE[0] <= ord(c) <= DEVA_RANGE[1])
        return (deva / total) >= 0.25

    if lang == "guj":
        gujr = sum(1 for c in letters if GUJR_RANGE[0] <= ord(c) <= GUJR_RANGE[1])
        return (gujr / total) >= 0.25

    return True


# -----------------------------------------------------------------------------
# Parallel-pair filters
# -----------------------------------------------------------------------------

MIN_WORDS = 2
MAX_WORDS = 120
MAX_LEN_RATIO = 3.0      # max(words_a, words_b) / min(words_a, words_b)


def keep_pair(a: str, b: str, lang_a: str, lang_b: str) -> bool:
    if not a or not b:
        return False
    if a == b:                                   # untranslated copy
        return False
    if not (is_valid_line(a, lang_a) and is_valid_line(b, lang_b)):
        return False

    na, nb = len(a.split()), len(b.split())

    if na < MIN_WORDS or nb < MIN_WORDS:
        return False
    if na > MAX_WORDS or nb > MAX_WORDS:
        return False
    if max(na, nb) / min(na, nb) > MAX_LEN_RATIO:
        return False

    return True


def filter_parallel_pair(corpus_a, corpus_b, lang_a, lang_b):
    out_a, out_b = [], []
    for a, b in zip(corpus_a, corpus_b):
        if keep_pair(a, b, lang_a, lang_b):
            out_a.append(a)
            out_b.append(b)
    return out_a, out_b


clean_doi, clean_pan = filter_parallel_pair(clean["doi"], clean["pan"], "doi", "pan")
clean_guj, clean_mar = filter_parallel_pair(clean["guj"], clean["mar"], "guj", "mar")

print()
print("=" * 80)
print("Post-cleaning pair counts")
print("=" * 80)
print(f"guj-mar : {len(clean_guj):,} / {len(raw['guj']):,} kept")
print(f"doi-pan : {len(clean_doi):,} / {len(raw['doi']):,} kept")

# Quick eyeball check of the data
print("\nSample guj-mar pair:\n ", clean_guj[0], "\n ", clean_mar[0])
print("\nSample doi-pan pair:\n ", clean_doi[0], "\n ", clean_pan[0])


doi: 28,000 raw lines
pan: 28,000 raw lines
guj: 28,000 raw lines
mar: 28,000 raw lines

Post-cleaning pair counts
guj-mar : 27,999 / 28,000 kept
doi-pan : 27,970 / 28,000 kept

Sample guj-mar pair:
  ખાડી અને જળસંધિ વચ્ચેનો તફાવત સમજાવો. 
  आखात आणि सामुद्रधुनीमधील फरक स्पष्ट करा.

Sample doi-pan pair:
  बीऽ गी घट्ट डुंह्गेआई च राहओ। 
  ਬੀਜਾਂ ਨੂੰ ਘੱਟ ਡੂੰਘਾਈ 'ਤੇ ਬੀਜੋ।


In [4]:
# -----------------------------------------------------------------------------
# CELL 5 — Semantic Filtering with LaBSE   (UPDATED + ENABLED)
#
# Changes vs. the original (which was commented out):
#   * ENABLED. Misaligned / noisy pairs are the #1 reason a LoRA cannot beat
#     a strong pretrained model.
#   * Uses normalised embeddings + a row-wise dot product. The old
#     util.cos_sim(...).diag() builds an N x N matrix (28k x 28k ~ 3 GB) and
#     would run out of memory.
#   * Threshold is adaptive: max(SIM_FLOOR, bottom DROP_BOTTOM_PCT percentile).
#   * Dropped pairs are saved to CSV so you can inspect them.
#   * If LaBSE cannot be loaded the pipeline continues without it.
# -----------------------------------------------------------------------------

DROPPED_DIR = SPLIT_DIR / "dropped_by_labse"
DROPPED_DIR.mkdir(parents=True, exist_ok=True)


def filter_by_semantic_similarity(src_list, tgt_list, name, embed_model):

    print(f"\n[{name}] encoding {len(src_list):,} pairs with LaBSE ...")

    src_emb = embed_model.encode(
        src_list,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    tgt_emb = embed_model.encode(
        tgt_list,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    sims = (src_emb * tgt_emb).sum(axis=1)          # row-wise cosine

    threshold = max(SIM_FLOOR, float(np.percentile(sims, DROP_BOTTOM_PCT)))

    print(
        f"[{name}] similarity  min={sims.min():.3f}  "
        f"p5={np.percentile(sims, 5):.3f}  "
        f"median={np.median(sims):.3f}  max={sims.max():.3f}"
    )
    print(f"[{name}] threshold used = {threshold:.3f}")

    keep = sims >= threshold

    kept_src = [s for s, k in zip(src_list, keep) if k]
    kept_tgt = [t for t, k in zip(tgt_list, keep) if k]

    pd.DataFrame({
        "src": [s for s, k in zip(src_list, keep) if not k],
        "tgt": [t for t, k in zip(tgt_list, keep) if not k],
        "sim": sims[~keep],
    }).to_csv(DROPPED_DIR / f"{name}_dropped.csv", index=False)

    print(
        f"[{name}] kept {len(kept_src):,}, "
        f"dropped {int((~keep).sum()):,}"
    )

    return kept_src, kept_tgt


try:
    from sentence_transformers import SentenceTransformer

    print("Loading LaBSE quality-filter model ...")
    embed_model = SentenceTransformer(
        "sentence-transformers/LaBSE",
        device=str(DEVICE),
    )

    clean_guj, clean_mar = filter_by_semantic_similarity(
        clean_guj, clean_mar, "guj_mar", embed_model
    )
    clean_doi, clean_pan = filter_by_semantic_similarity(
        clean_doi, clean_pan, "doi_pan", embed_model
    )

    # Free VRAM before IndicTrans2 is loaded
    del embed_model

except Exception as e:
    print("\n!! LaBSE filtering skipped:", repr(e))
    print("!! Continuing with the rule-based filtered data only.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nFinal pair counts after all filtering:")
print(f"  guj-mar : {len(clean_guj):,}")
print(f"  doi-pan : {len(clean_doi):,}")


Loading LaBSE quality-filter model ...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


... (13 outputs left out)

[guj_mar] similarity  min=0.303  p5=0.844  median=0.940  max=0.995
[guj_mar] threshold used = 0.844
[guj_mar] kept 26,599, dropped 1,400

[doi_pan] encoding 27,970 pairs with LaBSE ...


Batches:   0%|          | 0/219 [00:00<?, ?it/s]

Batches:   0%|          | 0/219 [00:00<?, ?it/s]

[doi_pan] similarity  min=-0.033  p5=0.403  median=0.716  max=0.988
[doi_pan] threshold used = 0.550
[doi_pan] kept 23,172, dropped 4,798

Final pair counts after all filtering:
  guj-mar : 26,599
  doi-pan : 23,172


In [5]:
# -----------------------------------------------------------------------------
# CELL 6 — Dataset Splitting   (UPDATED)
#
# Changes vs. the original (70/15/15):
#   * 90 / 5 / 5 split (VAL_FRAC / TEST_FRAC in the config cell): the model
#     now trains on ~90% of the cleaned data instead of 70%.
#   * De-duplicates by SOURCE and by TARGET separately, so the same sentence
#     can never appear in both train and test (leakage inflates / distorts
#     the base-vs-LoRA comparison).
#   * The fixed 1,000-sentence validation subset, the test set and all
#     variable names used by later cells are unchanged.
#
# NOTE: for the FINAL competition submission, set VAL_FRAC=0.02, TEST_FRAC=0.0
# (or retrain on train+val+test with the epoch count found here).
# -----------------------------------------------------------------------------

import pandas as pd
from sklearn.model_selection import train_test_split


def make_pair_df(col_a, col_b, name_a, name_b):
    df = pd.DataFrame({name_a: col_a, name_b: col_b})
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset=name_a)
    df = df.drop_duplicates(subset=name_b)
    return df.reset_index(drop=True)


df_guj_mar = make_pair_df(clean_guj, clean_mar, "guj", "mar")
df_doi_pan = make_pair_df(clean_doi, clean_pan, "doi", "pan")


def split_train_val_test(df, seed=SEED):

    holdout = VAL_FRAC + TEST_FRAC

    train, temp = train_test_split(
        df,
        test_size=holdout,
        random_state=seed,
        shuffle=True,
    )

    if TEST_FRAC <= 0:
        return (
            train.reset_index(drop=True),
            temp.reset_index(drop=True),
            temp.iloc[:0].reset_index(drop=True),
        )

    val, test = train_test_split(
        temp,
        test_size=TEST_FRAC / holdout,
        random_state=seed + 1,
        shuffle=True,
    )

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


gm_train, gm_val, gm_test = split_train_val_test(df_guj_mar)
dp_train, dp_val, dp_test = split_train_val_test(df_doi_pan)


# -----------------------------------------------------------------------------
# Split statistics
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print(f"DATASET SPLIT  (val={VAL_FRAC:.0%}, test={TEST_FRAC:.0%})")
print("=" * 80)

for title, total, tr, va, te in [
    ("Gujarati-Marathi", df_guj_mar, gm_train, gm_val, gm_test),
    ("Dogri-Punjabi", df_doi_pan, dp_train, dp_val, dp_test),
]:
    print(
        f"\n{title}"
        f"\n  Total      : {len(total):,}"
        f"\n  Train      : {len(tr):,} ({len(tr) / len(total):.1%})"
        f"\n  Validation : {len(va):,}"
        f"\n  Test       : {len(te):,}"
    )

expected_directional_train = 2 * (len(gm_train) + len(dp_train))
print("\n" + "-" * 80)
print(f"Expected directional training examples: {expected_directional_train:,}")
print("-" * 80)


# -----------------------------------------------------------------------------
# Save full splits
# -----------------------------------------------------------------------------

gm_train.to_csv(SPLIT_DIR / "guj_mar_train.csv", index=False)
gm_val.to_csv(SPLIT_DIR / "guj_mar_val.csv", index=False)
gm_test.to_csv(SPLIT_DIR / "guj_mar_test.csv", index=False)

dp_train.to_csv(SPLIT_DIR / "doi_pan_train.csv", index=False)
dp_val.to_csv(SPLIT_DIR / "doi_pan_val.csv", index=False)
dp_test.to_csv(SPLIT_DIR / "doi_pan_test.csv", index=False)


# -----------------------------------------------------------------------------
# Direction-specific dataframes
# -----------------------------------------------------------------------------

def make_direction_df(pair_df, src_col, tgt_col):
    return pd.DataFrame({
        "src_text": pair_df[src_col].tolist(),
        "tgt_text": pair_df[tgt_col].tolist(),
    })


DIRECTIONS = ["guj2mar", "mar2guj", "doi2pan", "pan2doi"]

_DIRECTION_SPEC = {
    "guj2mar": ("guj", "mar", (gm_train, gm_val, gm_test)),
    "mar2guj": ("mar", "guj", (gm_train, gm_val, gm_test)),
    "doi2pan": ("doi", "pan", (dp_train, dp_val, dp_test)),
    "pan2doi": ("pan", "doi", (dp_train, dp_val, dp_test)),
}

direction_data = {}

for direction, (src, tgt, (tr, va, te)) in _DIRECTION_SPEC.items():
    direction_data[direction] = {
        "src_lang": LANG_TAG[src],
        "tgt_lang": LANG_TAG[tgt],
        "train": make_direction_df(tr, src, tgt),
        "val": make_direction_df(va, src, tgt),
        "test": make_direction_df(te, src, tgt),
    }


# -----------------------------------------------------------------------------
# Fixed validation subset used for per-epoch checkpoint selection
# -----------------------------------------------------------------------------

VALIDATION_SELECTION_SIZE = 1000

for direction in DIRECTIONS:

    full_val = direction_data[direction]["val"]
    n_select = min(VALIDATION_SELECTION_SIZE, len(full_val))

    fixed_val = (
        full_val
        .sample(n=n_select, random_state=SEED)
        .reset_index(drop=True)
    )

    direction_data[direction]["val_selection"] = fixed_val

    fixed_val.to_csv(
        SPLIT_DIR / f"{direction}_val_selection_{n_select}.csv",
        index=False,
    )


print("\n" + "=" * 80)
print("FINAL DATASET CONFIGURATION")
print("=" * 80)

for direction in DIRECTIONS:
    d = direction_data[direction]
    print(
        f"\n{direction}:"
        f"\n  Train          : {len(d['train']):,}"
        f"\n  Full validation: {len(d['val']):,}"
        f"\n  Epoch val      : {len(d['val_selection']):,}"
        f"\n  Test           : {len(d['test']):,}"
    )

print("\n✓ Splits created (duplicate-free across splits).")
print("✓ Fixed validation-selection subsets created.")
print("✓ Test sets remain untouched.")



DATASET SPLIT  (val=5%, test=5%)

Gujarati-Marathi
  Total      : 26,572
  Train      : 23,914 (90.0%)
  Validation : 1,329
  Test       : 1,329

Dogri-Punjabi
  Total      : 23,151
  Train      : 20,835 (90.0%)
  Validation : 1,158
  Test       : 1,158

--------------------------------------------------------------------------------
Expected directional training examples: 89,498
--------------------------------------------------------------------------------

FINAL DATASET CONFIGURATION

guj2mar:
  Train          : 23,914
  Full validation: 1,329
  Epoch val      : 1,000
  Test           : 1,329

mar2guj:
  Train          : 23,914
  Full validation: 1,329
  Epoch val      : 1,000
  Test           : 1,329

doi2pan:
  Train          : 20,835
  Full validation: 1,158
  Epoch val      : 1,000
  Test           : 1,158

pan2doi:
  Train          : 20,835
  Full validation: 1,158
  Epoch val      : 1,000
  Test           : 1,158

✓ Splits created (duplicate-free across splits).
✓ Fixed vali

In [6]:
# -----------------------------------------------------------------------------
# 6. Load Base Model and Tokenizer
# -----------------------------------------------------------------------------
# IMPORTANT:
# We do NOT attach a single shared LoRA adapter here.
#
# We will train:
#   1. Gujarati <-> Marathi adapter
#   2. Dogri    <-> Punjabi adapter
#
# Each adapter starts from a fresh copy of the original IndicTrans2 base model.
# -----------------------------------------------------------------------------

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
)

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient


# -----------------------------------------------------------------------------
# Free previous models
# -----------------------------------------------------------------------------

if "base_model" in globals():
    del base_model

if "peft_model" in globals():
    del peft_model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Hugging Face authentication
# -----------------------------------------------------------------------------

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", None)


# -----------------------------------------------------------------------------
# Tokenizer
# -----------------------------------------------------------------------------

print("Loading IndicTrans2 tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN,
)

print("Tokenizer loaded successfully.")


# -----------------------------------------------------------------------------
# Load BASE model
# -----------------------------------------------------------------------------

print("\nLoading IndicTrans2 Indic-Indic dist-320M base model...")

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=MODEL_DTYPE,
    token=HF_TOKEN,
).to(DEVICE)

base_model.eval()

print("Base model loaded successfully.")


# -----------------------------------------------------------------------------
# No LoRA is attached here.
# -----------------------------------------------------------------------------

print("\n" + "=" * 75)
print("BASE MODEL READY")
print("=" * 75)

print(f"Model : {MODEL_NAME}")
print(f"Device: {DEVICE}")
print("LoRA : NOT ATTACHED")
print("=" * 75)

Loading IndicTrans2 tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-dist-320M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Base model loaded successfully.

BASE MODEL READY
Model : ai4bharat/indictrans2-indic-indic-dist-320M
Device: cuda
LoRA : NOT ATTACHED


In [7]:
# =============================================================================
# CELL 7/8 — DIRECTION-SPECIFIC TOKENIZATION   (UPDATED)
#
# Changes vs. the original:
#   * Training inputs are produced with IndicProcessor(inference=True) -- the
#     SAME pre-processing used at evaluation / test time, so train and test
#     inputs are guaranteed to look identical.
#   * Pairs that would be TRUNCATED (>= MAX_LENGTH tokens) are dropped instead
#     of being silently cut (a truncated target teaches the model to stop early).
#   * Sanity checks: tagged source, token ids, EOS at the end of the labels.
# =============================================================================

import gc
import pandas as pd
import datasets as hf_datasets

import torch
from IndicTransToolkit import IndicProcessor
from transformers import AutoConfig


MAX_LENGTH = 256

# inference=False -> no <ID1> placeholder queue is filled during training.
ip_train = IndicProcessor(inference=False)


def build_direction_dataframe(direction_name, split="train"):

    info = direction_data[direction_name]

    df = info[split][["src_text", "tgt_text"]].reset_index(drop=True)

    if QUICK_TEST_MODE:
        df = df.iloc[:500].copy()

    tagged_source = ip_train.preprocess_batch(
        df["src_text"].tolist(),
        src_lang=info["src_lang"],
        tgt_lang=info["tgt_lang"],
    )

    # -------------------------------------------------------------------------
    # ROOT-CAUSE FIX: the TARGET must go through the same IndicProcessor
    # pipeline (normalise digits, tokenise, TRANSLITERATE TO DEVANAGARI) as
    # the source. IndicTrans2 generates Devanagari and postprocess_batch()
    # converts it back to Gujarati / Gurmukhi. Raw Gujarati / Gurmukhi labels
    # are out-of-vocabulary for the decoder -> loss ~11 -> model destroyed.
    # (Marathi / Dogri targets are already Devanagari, which is why only
    # guj2mar and pan2doi worked before.)
    # -------------------------------------------------------------------------
    tagged_target = ip_train.preprocess_batch(
        df["tgt_text"].tolist(),
        src_lang=info["tgt_lang"],      # script of the TARGET text
        tgt_lang=None,
        is_target=True,                 # no language tags are added
    )

    return pd.DataFrame({
        "src_tagged": tagged_source,
        "tgt_text": tagged_target,
    })


def tokenize_direction_batch(batch):

    inputs = tokenizer(
        batch["src_tagged"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    labels = tokenizer(
        text_target=batch["tgt_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs["labels"] = labels["input_ids"]

    return inputs


def not_truncated(example):
    return (
        len(example["input_ids"]) < MAX_LENGTH
        and len(example["labels"]) < MAX_LENGTH
    )



# -----------------------------------------------------------------------------
# CUSTOM COLLATOR  (THE FIX FOR: "You have to specify either decoder_input_ids
# or decoder_inputs_embeds")
#
# The IndicTrans2 remote-code model does NOT build decoder_input_ids from
# `labels` by itself (BART / T5 do). DataCollatorForSeq2Seq only does it when
# it is given the model, and even then relies on a method this model lacks.
# So we build them explicitly:  decoder_input = [decoder_start] + labels[:-1]
# -----------------------------------------------------------------------------

_cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else _cfg.pad_token_id
DECODER_START_ID = (
    _cfg.decoder_start_token_id
    if getattr(_cfg, "decoder_start_token_id", None) is not None
    else tokenizer.eos_token_id
)

print(f"pad_token_id={PAD_ID} | decoder_start_token_id={DECODER_START_ID} "
      f"| eos_token_id={tokenizer.eos_token_id}")


class IndicSeq2SeqCollator:

    def __init__(self, tokenizer, pad_id, decoder_start_id,
                 label_pad_id=-100, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_id = pad_id
        self.decoder_start_id = decoder_start_id
        self.label_pad_id = label_pad_id
        self.pad_to_multiple_of = pad_to_multiple_of

    def _round_up(self, n):
        m = self.pad_to_multiple_of
        return n if not m else ((n + m - 1) // m) * m

    def __call__(self, features):

        labels = [list(f["labels"]) for f in features]

        # ---- encoder side (input_ids + attention_mask) ----
        enc_features = [
            {k: v for k, v in f.items() if k in ("input_ids", "attention_mask")}
            for f in features
        ]
        batch = self.tokenizer.pad(
            enc_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # ---- labels (right padded with -100) ----
        max_len = self._round_up(max(len(l) for l in labels))
        label_ids = torch.full((len(labels), max_len), self.label_pad_id, dtype=torch.long)
        for i, l in enumerate(labels):
            label_ids[i, :len(l)] = torch.tensor(l, dtype=torch.long)

        # ---- decoder inputs = shift-right(labels) ----
        decoder_input_ids = torch.full_like(label_ids, self.pad_id)
        decoder_input_ids[:, 1:] = label_ids[:, :-1]
        decoder_input_ids[:, 0] = self.decoder_start_id
        decoder_input_ids = decoder_input_ids.masked_fill(
            decoder_input_ids == self.label_pad_id, self.pad_id
        )

        batch["labels"] = label_ids
        batch["decoder_input_ids"] = decoder_input_ids

        return batch


direction_train_tok = {}
direction_val_tok = {}
direction_data_collators = {}

for direction in DIRECTIONS:

    print("\n" + "=" * 80)
    print(f"PREPARING {direction}")
    print("=" * 80)

    train_df = build_direction_dataframe(direction, split="train")
    val_df = build_direction_dataframe(direction, split="val")

    print(f"Raw train examples: {len(train_df):,}")
    print(f"Raw val examples  : {len(val_df):,}")

    train_ds = hf_datasets.Dataset.from_pandas(train_df, preserve_index=False)
    val_ds = hf_datasets.Dataset.from_pandas(val_df, preserve_index=False)

    train_tok = train_ds.map(
        tokenize_direction_batch,
        batched=True,
        remove_columns=["src_tagged", "tgt_text"],
        desc=f"Tokenizing {direction} train",
    )

    val_tok = val_ds.map(
        tokenize_direction_batch,
        batched=True,
        remove_columns=["src_tagged", "tgt_text"],
        desc=f"Tokenizing {direction} validation",
    )

    n_before = len(train_tok)
    train_tok = train_tok.filter(not_truncated)
    print(f"Dropped {n_before - len(train_tok):,} over-long train pairs")

    if MAX_TRAIN_PER_DIRECTION is not None and len(train_tok) > MAX_TRAIN_PER_DIRECTION:
        train_tok = train_tok.shuffle(seed=SEED).select(range(MAX_TRAIN_PER_DIRECTION))
        print(f"Capped training set to {len(train_tok):,} examples")

    direction_train_tok[direction] = train_tok
    direction_val_tok[direction] = val_tok

    direction_data_collators[direction] = IndicSeq2SeqCollator(
        tokenizer=tokenizer,
        pad_id=PAD_ID,
        decoder_start_id=DECODER_START_ID,
    )


print("\n" + "=" * 80)
print("DIRECTIONAL TOKENIZATION COMPLETE")
print("=" * 80)

for direction in DIRECTIONS:
    print(
        f"{direction:10s} | "
        f"train={len(direction_train_tok[direction]):,} | "
        f"val={len(direction_val_tok[direction]):,}"
    )


# =============================================================================
# SANITY CHECK  -- LOOK AT THIS OUTPUT BEFORE TRAINING
# =============================================================================

print("\n" + "=" * 80)
print("LANGUAGE / TOKEN SANITY CHECK")
print("=" * 80)

for direction in DIRECTIONS:

    info = direction_data[direction]
    row = build_direction_dataframe(direction, split="train").iloc[0]
    ex = direction_train_tok[direction][0]

    print("\nDirection       :", direction)
    print("Source language :", info["src_lang"])
    print("Target language :", info["tgt_lang"])
    print("Tagged source   :", row["src_tagged"][:200])
    print("Target          :", row["tgt_text"][:200])
    print("Src tokens (head):", tokenizer.convert_ids_to_tokens(ex["input_ids"])[:6])
    print("Label last id   :", ex["labels"][-1], "(eos =", tokenizer.eos_token_id, ")")
    deva = sum(1 for ch in row["tgt_text"] if 0x0900 <= ord(ch) <= 0x097F)
    alpha = sum(1 for ch in row["tgt_text"] if ch.isalpha())
    print(f"Target Devanagari share: {deva / max(alpha, 1):.0%}  (must be ~100%)")
    assert deva / max(alpha, 1) > 0.9, (
        f"{direction}: target is NOT in Devanagari -> labels would be wrong!"
    )
    print("Label decode    :", tokenizer.decode(ex["labels"], skip_special_tokens=True)[:200])

print("\nExpect: source starts with '<src_tag> <tgt_tag>', both Source and Target are")
print("Devanagari for EVERY direction, and the last label id equals eos.")


# -----------------------------------------------------------------------------
# COLLATOR SANITY CHECK  (decoder_input_ids must exist and be shifted labels)
# -----------------------------------------------------------------------------

_demo = direction_data_collators[DIRECTIONS[0]](
    [direction_train_tok[DIRECTIONS[0]][i] for i in range(2)]
)
print("\nCollator keys      :", list(_demo.keys()))
print("labels[0][:8]      :", _demo["labels"][0][:8].tolist())
print("decoder_input[0][:8]:", _demo["decoder_input_ids"][0][:8].tolist())
assert "decoder_input_ids" in _demo, "decoder_input_ids missing!"
assert _demo["decoder_input_ids"][0, 0].item() == DECODER_START_ID
assert (_demo["decoder_input_ids"][0, 1:4] == _demo["labels"][0, 0:3]).all()
print("✓ Collator OK: decoder_input_ids = [decoder_start] + labels[:-1]")


pad_token_id=1 | decoder_start_token_id=2 | eos_token_id=2

PREPARING guj2mar
Raw train examples: 23,914
Raw val examples  : 1,329


Tokenizing guj2mar train:   0%|          | 0/23914 [00:00<?, ? examples/s]

Tokenizing guj2mar validation:   0%|          | 0/1329 [00:00<?, ? examples/s]

Filter:   0%|          | 0/23914 [00:00<?, ? examples/s]

Dropped 0 over-long train pairs

PREPARING mar2guj
Raw train examples: 23,914
Raw val examples  : 1,329


Tokenizing mar2guj train:   0%|          | 0/23914 [00:00<?, ? examples/s]

... (7 outputs left out)

Tokenizing pan2doi train:   0%|          | 0/20835 [00:00<?, ? examples/s]

Tokenizing pan2doi validation:   0%|          | 0/1158 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20835 [00:00<?, ? examples/s]

Dropped 0 over-long train pairs

DIRECTIONAL TOKENIZATION COMPLETE
guj2mar    | train=23,914 | val=1,329
mar2guj    | train=23,914 | val=1,329
doi2pan    | train=20,835 | val=1,158
pan2doi    | train=20,835 | val=1,158

LANGUAGE / TOKEN SANITY CHECK

Direction       : guj2mar
Source language : guj_Gujr
Target language : mar_Deva
Tagged source   : guj_Gujr mar_Deva स्वच्छ सर्वेक्षण , 2021मां राज्यनी 18 स्थानिक संस्थाओने विविध श्रेणीओमां पुरस्कार आपवामां आव्या .
Target          : स्वच्छ सर्वेक्षण , 2021 मध्ये प्रदेशातील 18 स्थानिक स्वराज्य संस्थांना विविध श्रेणींमध्ये पुरस्कृत केले गेले .
Src tokens (head): ['guj_Gujr', 'mar_Deva', '▁स्वच्छ', '▁सर्वेक्षण', '▁,', '▁2021']
Label last id   : 2 (eos = 2 )
Target Devanagari share: 174%  (must be ~100%)
Label decode    : स्वच्छ सर्वेक्षण, 2021 मध्ये प्रदेशातील 18 स्थानिक स्वराज्य संस्थांना विविध श्रेणींमध्ये पुरस्कृत केले गेले.

Direction       : mar2guj
Source language : mar_Deva
Target language : guj_Gujr
Tagged source   : mar_Deva guj_Gujr 

In [8]:
# =============================================================================
# PRE-FLIGHT CHECK  (takes ~1 minute, runs BEFORE any training)
#
# Measures the loss of the *pretrained* model on the labels of all four
# directions. A healthy pipeline gives ~2-5 for every direction. The previous
# run gave 3.2 / 3.5 for the Devanagari targets and 10-11 for Gujarati and
# Gurmukhi targets -- which is exactly what this cell is meant to catch
# up-front instead of after 5 hours of training.
# =============================================================================

base_model.eval()

preflight = {}

for direction in DIRECTIONS:

    ds = direction_train_tok[direction]
    batch = direction_data_collators[direction]([ds[i] for i in range(16)])
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=MODEL_DTYPE):
        preflight[direction] = base_model(**batch).loss.item()

    print(f"{direction:8s} pretrained-model loss on its labels = {preflight[direction]:6.3f}")

bad = {d: round(l, 2) for d, l in preflight.items() if l > 8}

if bad:
    raise RuntimeError(
        f"PRE-FLIGHT FAILED: {bad}. Labels for these directions do not match "
        "what the pretrained decoder expects. Do NOT start training."
    )

print("\n✓ PRE-FLIGHT PASSED: all four directions have sane label loss.")


guj2mar  pretrained-model loss on its labels =  1.436
mar2guj  pretrained-model loss on its labels =  1.163
doi2pan  pretrained-model loss on its labels =  1.005
pan2doi  pretrained-model loss on its labels =  1.840

✓ PRE-FLIGHT PASSED: all four directions have sane label loss.


In [9]:
# =============================================================================
# CELL 8 — TRANSLATION + BLEU / chrF++ EVALUATION
#
# Evaluation metrics:
#   1. BLEU
#   2. chrF++
#
# Beam size = 5.
#
# These are used consistently for:
#   - Base-model validation
#   - LoRA validation
#   - Checkpoint selection
#   - Final test evaluation
#   - Base vs LoRA comparison
# =============================================================================

import gc
import pandas as pd
import torch

from tqdm.auto import tqdm


# =============================================================================
# SACREBLEU CONFIGURATION
# =============================================================================

CHRFPP = sacrebleu.CHRF(
    word_order=2
)


# -----------------------------------------------------------------------------
# BLEU tokenizer
# -----------------------------------------------------------------------------

try:

    sacrebleu.corpus_bleu(
        ["test"],
        [["test"]],
        tokenize="flores200",
    )

    BLEU_TOKENIZE = "flores200"

except Exception:

    BLEU_TOKENIZE = "13a"


print(
    "BLEU tokenizer:",
    BLEU_TOKENIZE
)


# =============================================================================
# INDIC PROCESSOR
# =============================================================================

ip_eval = IndicProcessor(
    inference=True
)


# =============================================================================
# TRANSLATION
# =============================================================================

@torch.inference_mode()
def translate(
    model,
    sentences,
    src_lang,
    tgt_lang,
    batch_size=16,
    num_beams=5,
    max_new_tokens=200,
):

    model.eval()

    outputs = []

    for i in tqdm(
        range(
            0,
            len(sentences),
            batch_size,
        ),
        desc=(
            f"Translating "
            f"{src_lang.split('_')[0]} → "
            f"{tgt_lang.split('_')[0]}"
        ),
    ):

        batch = sentences[
            i:i + batch_size
        ]

        processed = ip_eval.preprocess_batch(
            batch,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )

        inputs = tokenizer(
            processed,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(DEVICE)

        generated_tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
            use_cache=True,
        )

        decoded = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        decoded = ip_eval.postprocess_batch(
            decoded,
            lang=tgt_lang,
        )

        outputs.extend(
            decoded
        )

    return outputs


# =============================================================================
# BLEU + chrF++
# =============================================================================

def compute_lexical_metrics(
    predictions,
    references,
):

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references],
        tokenize=BLEU_TOKENIZE,
    )

    chrfpp = CHRFPP.corpus_score(
        predictions,
        [references],
    )

    return {
        "BLEU": bleu.score,
        "chrF++": chrfpp.score,
    }


# =============================================================================
# COMPLETE EVALUATION
# =============================================================================

def evaluate_direction(
    model,
    df,
    src_lang,
    tgt_lang,
    direction_name,
    max_examples=None,
):

    if max_examples is not None:

        df_eval = (
            df.iloc[:max_examples]
            .copy()
        )

    else:

        df_eval = df.copy()


    sources = df_eval[
        "src_text"
    ].tolist()

    references = df_eval[
        "tgt_text"
    ].tolist()


    # -------------------------------------------------------------------------
    # Generate translations
    # -------------------------------------------------------------------------

    predictions = translate(
        model=model,
        sentences=sources,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        num_beams=5,
        max_new_tokens=200,
    )


    # -------------------------------------------------------------------------
    # BLEU + chrF++
    # -------------------------------------------------------------------------

    metrics = compute_lexical_metrics(
        predictions,
        references,
    )


    metrics.update({

        "direction":
            direction_name,

        "n":
            len(predictions),

        "predictions":
            predictions,

        "references":
            references,

    })


    return metrics


print("\n")
print("=" * 80)
print("BLEU + chrF++ EVALUATION PIPELINE READY")
print("=" * 80)

print(
    "Metrics: BLEU | chrF++"
)

print(
    "Checkpoint selection: validation chrF++"
)

print(
    "Decoding: beam size = 5"
)

BLEU tokenizer: flores200


BLEU + chrF++ EVALUATION PIPELINE READY
Metrics: BLEU | chrF++
Checkpoint selection: validation chrF++
Decoding: beam size = 5


In [10]:
# =============================================================================
# CELL 9 — BASE MODEL VALIDATION BENCHMARK
#
# The base model is evaluated on the SAME fixed 1,000-example validation
# subset used for epoch-level LoRA checkpoint selection.
#
# Metrics:
#   BLEU
#   chrF++
#
# The full validation set is NOT consumed here.
# It will be evaluated after LoRA training.
# =============================================================================

EVAL_SUBSAMPLE = (
    50
    if QUICK_TEST_MODE
    else VALIDATION_SELECTION_SIZE
)


base_val_results = []


print("=" * 90)
print(
    "BASE IndicTrans2 — FIXED VALIDATION SUBSET"
)
print("=" * 90)

print(
    f"Examples per direction: "
    f"{EVAL_SUBSAMPLE:,}"
)


for direction in DIRECTIONS:

    info = direction_data[
        direction
    ]


    # -------------------------------------------------------------------------
    # Use the exact fixed validation subset
    # -------------------------------------------------------------------------

    val_df = info[
        "val_selection"
    ]


    result = evaluate_direction(

        model=base_model,

        df=val_df,

        src_lang=info["src_lang"],

        tgt_lang=info["tgt_lang"],

        direction_name=direction,

        max_examples=None,
    )


    base_val_results.append(
        result
    )


    print(
        f"\n[{direction}]"
    )

    print(
        f"BLEU   = "
        f"{result['BLEU']:.2f}"
    )

    print(
        f"chrF++ = "
        f"{result['chrF++']:.2f}"
    )


# =============================================================================
# RESULTS TABLE
# =============================================================================

base_val_table = pd.DataFrame([

    {
        "Direction":
            r["direction"],

        "N":
            r["n"],

        "BLEU":
            r["BLEU"],

        "chrF++":
            r["chrF++"],
    }

    for r in base_val_results

])


print("\n")
print("=" * 90)
print(
    "BASE IndicTrans2 — FIXED VALIDATION RESULTS"
)
print("=" * 90)


display(
    base_val_table.round(4)
)


base_val_table.to_csv(

    RESULTS_DIR /
    "base_fixed_1000_validation_results.csv",

    index=False,
)


print(
    "\nSaved:",
    RESULTS_DIR /
    "base_fixed_1000_validation_results.csv"
)

BASE IndicTrans2 — FIXED VALIDATION SUBSET
Examples per direction: 1,000


Translating guj → mar:   0%|          | 0/63 [00:00<?, ?it/s]


[guj2mar]
BLEU   = 41.28
chrF++ = 58.60


Translating mar → guj:   0%|          | 0/63 [00:00<?, ?it/s]


[mar2guj]
BLEU   = 44.37
chrF++ = 59.17


Translating doi → pan:   0%|          | 0/63 [00:00<?, ?it/s]


[doi2pan]
BLEU   = 45.09
chrF++ = 58.58


Translating pan → doi:   0%|          | 0/63 [00:00<?, ?it/s]


[pan2doi]
BLEU   = 39.19
chrF++ = 53.18


BASE IndicTrans2 — FIXED VALIDATION RESULTS


,Direction,N,BLEU,chrF++
0,guj2mar,1000,41.2814,58.6045
1,mar2guj,1000,44.3745,59.1682
2,doi2pan,1000,45.0931,58.5768
3,pan2doi,1000,39.1937,53.1800



Saved: /kaggle/working/experiment1_indictrans2_lora/results/base_fixed_1000_validation_results.csv


In [11]:
# -----------------------------------------------------------------------------
# 10. FREE BASE MODEL BEFORE PAIR-SPECIFIC TRAINING
# -----------------------------------------------------------------------------

print("Freeing base IndicTrans2 model from GPU...")


if "base_model" in globals():
    del base_model


gc.collect()


if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("GPU memory cleared successfully.")

Freeing base IndicTrans2 model from GPU...
GPU memory cleared successfully.


In [12]:
# ============================================================
# CELL 11 — DIRECTION-SPECIFIC LoRA TRAINING   (UPDATED, dist-320M)
#
# Why the original did not beat the base model, and what changed:
#
#   1. LR 2e-5 is a full-fine-tuning LR. LoRA's B-matrices start at 0, so
#      the adapter barely moved in ~900 steps.      -> LR 2e-4, more epochs
#   2. r=16 on attention only.                       -> r=32 on attention AND
#                                                       FFN (fc1, fc2)
#   3. No label smoothing.                           -> 0.1
#   4. Random batches waste compute on padding.      -> group_by_length
#   5. Old checkpoints would be auto-resumed.        -> new output folder (v2)
#   6. No visibility into whether learning happens.  -> initial-loss sanity
#                                                       check, saved loss log
#   7. Runs all epochs even when nothing improves.   -> early stopping on
#                                                       validation chrF++
#
#   8. FIX: "You have to specify either decoder_input_ids or
#      decoder_inputs_embeds" -> IndicTrans2 does not shift labels itself, so
#      the custom collator (cell above) now supplies decoder_input_ids.
#
# Everything else (per-direction adapters, epoch-end generation-based
# validation, best-adapter-by-chrF++, resume from checkpoint) is preserved.
# ============================================================

import os
import gc
import json
import glob
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)

from peft import LoraConfig, get_peft_model


print("=" * 90)
print("DIRECTION-SPECIFIC LoRA TRAINING")
print("=" * 90)
print(f"Device        : {DEVICE}")
print(f"Model dtype   : {MODEL_DTYPE}")
print(f"Directions    : {DIRECTIONS}")
print(f"LoRA r/alpha  : {LORA_R}/{LORA_ALPHA}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Eff. batch    : {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"Output root   : {DIRECTION_OUTPUT_ROOT}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. Enable a Kaggle GPU accelerator.")

os.makedirs(DIRECTION_OUTPUT_ROOT, exist_ok=True)


# ============================================================
# LoRA CONFIGURATION
# ============================================================

LORA_CONFIG = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "out_proj",
        "fc1",
        "fc2",
    ],
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)


# ============================================================
# VALIDATION  (fixed subset, generation-based)
# ============================================================

def evaluate_validation_fast(
    model,
    tokenizer,
    processor,
    validation_data,
    src_lang,
    tgt_lang,
    batch_size=16,
    num_beams=VAL_BEAMS,
):

    model.eval()

    predictions = []
    references = []

    for start in range(0, len(validation_data), batch_size):

        batch = validation_data.iloc[start:start + batch_size]

        src_texts = batch["src_text"].tolist()      # IndicProcessor needs lists
        tgt_texts = batch["tgt_text"].tolist()

        processed = processor.preprocess_batch(
            src_texts,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
        )

        inputs = tokenizer(
            processed,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                num_beams=num_beams,
                max_new_tokens=200,
                early_stopping=True,
                use_cache=True,
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        decoded = processor.postprocess_batch(decoded, lang=tgt_lang)
        decoded = [x.strip() for x in decoded]

        predictions.extend(decoded)
        references.extend(tgt_texts)

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references],
        tokenize=BLEU_TOKENIZE,
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references],
        word_order=2,
    ).score

    model.train()

    return {"bleu": bleu, "chrf": chrf}


# ============================================================
# CHECKPOINT UTILITIES
# ============================================================

def get_latest_checkpoint(output_dir):

    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))

    if not checkpoints:
        return None

    def step(path):
        try:
            return int(os.path.basename(path).split("-")[-1])
        except ValueError:
            return -1

    return sorted(checkpoints, key=step)[-1]


def load_previous_best_chrf(output_dir):

    best_file = os.path.join(output_dir, "best_metric.json")

    if not os.path.exists(best_file):
        return -float("inf")

    try:
        with open(best_file, "r") as f:
            return float(json.load(f)["best_chrf"])
    except Exception:
        return -float("inf")


# ============================================================
# CALLBACK — runs at the end of every epoch
#   1. translate fixed validation subset
#   2. BLEU + chrF++
#   3. save epoch adapter
#   4. save best adapter (by chrF++)
#   5. early stopping
# ============================================================

class ChrFCheckpointCallback(TrainerCallback):

    def __init__(
        self,
        tokenizer,
        processor,
        validation_data,
        src_lang,
        tgt_lang,
        output_dir,
        base_chrf=None,
        patience=EARLY_STOP_PATIENCE,
    ):
        self.tokenizer = tokenizer
        self.processor = processor
        self.validation_data = validation_data
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.output_dir = output_dir
        self.patience = patience
        self.bad_epochs = 0
        self.base_chrf = base_chrf

        self.best_chrf = load_previous_best_chrf(output_dir)

        if self.best_chrf > -float("inf"):
            print(f"\nRestored previous best chrF++ = {self.best_chrf:.4f}")

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        epoch = int(round(state.epoch))

        print("\n" + "=" * 80)
        print(f"VALIDATION AFTER EPOCH {epoch}")
        print("=" * 80)

        metrics = evaluate_validation_fast(
            model=model,
            tokenizer=self.tokenizer,
            processor=self.processor,
            validation_data=self.validation_data,
            src_lang=self.src_lang,
            tgt_lang=self.tgt_lang,
        )

        print(f"\nBLEU   : {metrics['bleu']:.4f}")
        print(f"chrF++ : {metrics['chrf']:.4f}")

        if self.base_chrf is not None:
            print(f"(base model chrF++ at same beams: {self.base_chrf:.4f})")

        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)

        with open(
            os.path.join(self.output_dir, f"epoch_{epoch}_metrics.json"), "w"
        ) as f:
            json.dump(
                {
                    "epoch": epoch,
                    "BLEU": metrics["bleu"],
                    "chrF++": metrics["chrf"],
                },
                f,
                indent=2,
            )

        if metrics["chrf"] > self.best_chrf:

            self.best_chrf = metrics["chrf"]
            self.bad_epochs = 0

            best_dir = os.path.join(self.output_dir, "best_adapter")
            os.makedirs(best_dir, exist_ok=True)
            model.save_pretrained(best_dir)

            with open(os.path.join(self.output_dir, "best_metric.json"), "w") as f:
                json.dump(
                    {
                        "best_epoch": epoch,
                        "best_chrf": self.best_chrf,
                        "best_bleu": metrics["bleu"],
                    },
                    f,
                    indent=2,
                )

            print("\n✓ NEW BEST ADAPTER SAVED")
            print(f"  Epoch  : {epoch}")
            print(f"  BLEU   : {metrics['bleu']:.4f}")
            print(f"  chrF++ : {self.best_chrf:.4f}")

        else:

            self.bad_epochs += 1

            print("\nNo improvement.")
            print(f"Best chrF++ remains {self.best_chrf:.4f}")
            print(f"Epochs without improvement: {self.bad_epochs}/{self.patience}")

            if self.bad_epochs >= self.patience:
                print("Early stopping triggered.")
                control.should_training_stop = True

        return control


# ============================================================
# TRAIN EACH DIRECTION INDEPENDENTLY
# ============================================================

trained_adapters = {}

for direction in DIRECTIONS:

    print("\n\n" + "=" * 90)
    print(f"TRAINING DIRECTION: {direction}")
    print("=" * 90)

    src_lang = direction_data[direction]["src_lang"]
    tgt_lang = direction_data[direction]["tgt_lang"]

    print(f"Source language: {src_lang}")
    print(f"Target language: {tgt_lang}")

    train_dataset = direction_train_tok[direction]
    val_dataset = direction_data[direction]["val_selection"]
    if QUICK_TEST_MODE:
        val_dataset = val_dataset.iloc[:50].reset_index(drop=True)
    collator = direction_data_collators[direction]

    print(f"Training examples  : {len(train_dataset):,}")
    print(f"Validation examples: {len(val_dataset):,}")

    direction_output_dir = os.path.join(DIRECTION_OUTPUT_ROOT, direction)
    os.makedirs(direction_output_dir, exist_ok=True)

    # Skip a direction that already finished (safe re-run after a session cut)
    if os.path.exists(os.path.join(direction_output_dir, "FINISHED")):
        print(f"{direction} already finished -> skipping.")
        trained_adapters[direction] = os.path.join(
            direction_output_dir, "best_adapter"
        )
        continue

    latest_checkpoint = get_latest_checkpoint(direction_output_dir)

    if latest_checkpoint is not None:
        print("\n" + "!" * 80)
        print(f"EXISTING CHECKPOINT FOUND -> resuming from {latest_checkpoint}")
        print("!" * 80)
    else:
        print("\nNo previous checkpoint found. Starting from the base model.")

    # --------------------------------------------------------
    # Fresh base model + fresh LoRA
    # --------------------------------------------------------

    print(f"\nLoading fresh base model for {direction}...")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=MODEL_DTYPE,
        token=HF_TOKEN,
    ).to(DEVICE)

    model = get_peft_model(model, LORA_CONFIG)
    model.print_trainable_parameters()

    # LoRA weights in fp32 (stable optimisation), frozen base stays fp16/bf16
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.float()

    # --------------------------------------------------------
    # SANITY CHECK: initial loss on a few training examples.
    # With a good pretrained model this should be roughly 1-4.
    # A value > 8 means labels / decoder inputs are mis-wired.
    # --------------------------------------------------------

    probe_loss = None
    try:
        model.eval()
        probe = collator([train_dataset[i] for i in range(8)])
        probe = {k: v.to(DEVICE) for k, v in probe.items()}
        assert "decoder_input_ids" in probe, "collator did not create decoder_input_ids"
        with torch.no_grad(), torch.autocast(
            device_type="cuda", dtype=MODEL_DTYPE
        ):
            probe_loss = model(**probe).loss.item()
        print(f"\n[SANITY] initial training loss (8 examples) = {probe_loss:.3f}")
    except Exception as e:
        print("[SANITY] loss probe skipped:", repr(e))
    finally:
        model.train()

    # STOP instead of silently wasting hours on mis-wired labels
    if probe_loss is not None and probe_loss > 8:
        raise RuntimeError(
            f"[{direction}] initial loss {probe_loss:.2f} > 8 -> labels are wrong "
            "(not Devanagari-transliterated?). Aborting before training."
        )

    # --------------------------------------------------------
    # Callback
    # --------------------------------------------------------

    validation_callback = ChrFCheckpointCallback(
        tokenizer=tokenizer,
        processor=ip_eval,
        validation_data=val_dataset,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        output_dir=direction_output_dir,
    )

    # --------------------------------------------------------
    # Training arguments
    # --------------------------------------------------------

    use_bf16 = (MODEL_DTYPE == torch.bfloat16)

    training_args = Seq2SeqTrainingArguments(
        output_dir=direction_output_dir,

        num_train_epochs=1 if QUICK_TEST_MODE else NUM_EPOCHS,

        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,

        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        max_grad_norm=1.0,
        label_smoothing_factor=LABEL_SMOOTHING,

        optim="adamw_torch",

        bf16=use_bf16,
        fp16=not use_bf16,

        group_by_length=True,           # far less padding -> faster on a T4

        logging_steps=50,

        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,

        evaluation_strategy="no",       # validation is done in the callback
        predict_with_generate=False,

        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=collator,
        callbacks=[validation_callback],
    )

    # --------------------------------------------------------
    # Train / resume
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("RESUMING TRAINING" if latest_checkpoint else "STARTING FRESH TRAINING")
    print("=" * 80)

    if latest_checkpoint is not None:
        trainer.train(resume_from_checkpoint=latest_checkpoint)
    else:
        trainer.train()

    # Save the loss curve so you can plot / inspect learning
    with open(os.path.join(direction_output_dir, "log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    # --------------------------------------------------------
    # Verify best adapter
    # --------------------------------------------------------

    best_adapter_dir = os.path.join(direction_output_dir, "best_adapter")

    if not os.path.exists(best_adapter_dir):
        raise RuntimeError(
            "Training finished but best_adapter was not created:\n"
            f"{best_adapter_dir}"
        )

    open(os.path.join(direction_output_dir, "FINISHED"), "w").close()

    trained_adapters[direction] = best_adapter_dir

    print("\n" + "=" * 80)
    print(f"✓ FINISHED {direction}")
    print(f"Best adapter: {best_adapter_dir}")
    print("=" * 80)

    # --------------------------------------------------------
    # Free GPU memory
    # --------------------------------------------------------

    del trainer
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


print("\n\n" + "=" * 90)
print("ALL FOUR DIRECTIONS FINISHED")
print("=" * 90)

for direction, path in trained_adapters.items():
    print(f"{direction:10s} -> {path}")


DIRECTION-SPECIFIC LoRA TRAINING
Device        : cuda
Model dtype   : torch.float16
Directions    : ['guj2mar', 'mar2guj', 'doi2pan', 'pan2doi']
LoRA r/alpha  : 96/128
Learning rate : 0.0002
Epochs        : 5
Eff. batch    : 16
Output root   : /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit


TRAINING DIRECTION: guj2mar
Source language: guj_Gujr
Target language: mar_Deva
Training examples  : 23,914
Validation examples: 1,000

No previous checkpoint found. Starting from the base model.

Loading fresh base model for guj2mar...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


trainable params: 38,928,384 || all params: 359,789,568 || trainable%: 10.819764513016676

[SANITY] initial training loss (8 examples) = 1.735


/usr/local/lib/python3.13/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



STARTING FRESH TRAINING


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


... (89 outputs left out)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



VALIDATION AFTER EPOCH 5

BLEU   : 45.7130
chrF++ : 58.5562


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



No improvement.
Best chrF++ remains 58.6637
Epochs without improvement: 1/2

✓ FINISHED pan2doi
Best adapter: /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/pan2doi/best_adapter


ALL FOUR DIRECTIONS FINISHED
guj2mar    -> /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/guj2mar/best_adapter
mar2guj    -> /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/mar2guj/best_adapter
doi2pan    -> /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/doi2pan/best_adapter
pan2doi    -> /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/pan2doi/best_adapter


In [13]:
# =============================================================================
# NEW CELL — FULL VALIDATION OF BEST LoRA ADAPTERS
#
# IMPORTANT:
#   Epoch-level checkpoint selection used the fixed 1,000-example subset.
#
#   Now that training is finished, evaluate the selected best adapter on the
#   ENTIRE validation set (~4,199 examples per direction).
#
# Metrics:
#   BLEU
#   chrF++
#
# The TEST set is still completely untouched.
# =============================================================================

import os
import gc
import pandas as pd
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
)

from peft import (
    PeftModel,
)


FULL_VALIDATION_RESULTS = []


print("=" * 100)
print("FULL VALIDATION — BEST DIRECTION-SPECIFIC LoRA")
print("=" * 100)

print(
    "Using the complete validation set."
)


for direction in DIRECTIONS:

    print("\n")
    print("#" * 100)

    print(
        f"FULL VALIDATION: {direction}"
    )

    print("#" * 100)


    # =========================================================================
    # BEST ADAPTER
    # =========================================================================

    best_adapter_dir = os.path.join(

        DIRECTION_OUTPUT_ROOT,

        direction,

        "best_adapter",

    )


    if not os.path.exists(
        best_adapter_dir
    ):

        raise FileNotFoundError(

            f"Missing best adapter:\n"
            f"{best_adapter_dir}"

        )


    # =========================================================================
    # FRESH BASE MODEL
    # =========================================================================

    print(
        "\nLoading fresh base model..."
    )


    base_model = (

        AutoModelForSeq2SeqLM.from_pretrained(

            MODEL_NAME,

            trust_remote_code=True,

            torch_dtype=MODEL_DTYPE,

            token=HF_TOKEN,

        )

        .to(DEVICE)

    )


    # =========================================================================
    # LOAD BEST LoRA
    # =========================================================================

    model = PeftModel.from_pretrained(

        base_model,

        best_adapter_dir,

    )


    model.eval()


    # =========================================================================
    # FULL VALIDATION
    # =========================================================================

    info = direction_data[
        direction
    ]


    full_val_df = info[
        "val"
    ]


    print(
        f"\nFull validation examples: "
        f"{len(full_val_df):,}"
    )


    result = evaluate_direction(

        model=model,

        df=full_val_df,

        src_lang=info["src_lang"],

        tgt_lang=info["tgt_lang"],

        direction_name=direction,

        max_examples=None,

    )


    # =========================================================================
    # STORE RESULTS
    # =========================================================================

    FULL_VALIDATION_RESULTS.append({

        "direction":
            direction,

        "N":
            result["n"],

        "BLEU":
            result["BLEU"],

        "chrF++":
            result["chrF++"],

        "checkpoint":
            "best_validation_chrF++",

    })


    print("\nFULL VALIDATION RESULTS")

    print(
        f"BLEU   : "
        f"{result['BLEU']:.2f}"
    )

    print(
        f"chrF++ : "
        f"{result['chrF++']:.2f}"
    )


    # =========================================================================
    # CLEANUP
    # =========================================================================

    del model

    del base_model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# =============================================================================
# SAVE
# =============================================================================

full_validation_df = pd.DataFrame(
    FULL_VALIDATION_RESULTS
)


full_validation_path = os.path.join(

    OUTPUT_DIR,

    "FINAL_full_validation_results.csv",

)


full_validation_df.to_csv(

    full_validation_path,

    index=False,

)


# =============================================================================
# DISPLAY
# =============================================================================

print("\n")

print(
    "=" * 100
)

print(
    "FINAL FULL VALIDATION RESULTS"
)

print(
    "=" * 100
)


display(
    full_validation_df.round(4)
)


print(
    "\nSaved:",
    full_validation_path
)

FULL VALIDATION — BEST DIRECTION-SPECIFIC LoRA
Using the complete validation set.


####################################################################################################
FULL VALIDATION: guj2mar
####################################################################################################

Loading fresh base model...

Full validation examples: 1,329


Translating guj → mar:   0%|          | 0/84 [00:00<?, ?it/s]


FULL VALIDATION RESULTS
BLEU   : 47.82
chrF++ : 63.73


####################################################################################################
FULL VALIDATION: mar2guj
####################################################################################################

Loading fresh base model...

Full validation examples: 1,329


Translating mar → guj:   0%|          | 0/84 [00:00<?, ?it/s]


FULL VALIDATION RESULTS
BLEU   : 50.48
chrF++ : 64.28


####################################################################################################
FULL VALIDATION: doi2pan
####################################################################################################

Loading fresh base model...

Full validation examples: 1,158


Translating doi → pan:   0%|          | 0/73 [00:00<?, ?it/s]


FULL VALIDATION RESULTS
BLEU   : 52.78
chrF++ : 65.07


####################################################################################################
FULL VALIDATION: pan2doi
####################################################################################################

Loading fresh base model...

Full validation examples: 1,158


Translating pan → doi:   0%|          | 0/73 [00:00<?, ?it/s]


FULL VALIDATION RESULTS
BLEU   : 45.89
chrF++ : 58.61


FINAL FULL VALIDATION RESULTS


,direction,N,BLEU,chrF++,checkpoint
0,guj2mar,1329,47.8168,63.7313,best_validation_chrF++
1,mar2guj,1329,50.4819,64.2804,best_validation_chrF++
2,doi2pan,1158,52.7791,65.0728,best_validation_chrF++
3,pan2doi,1158,45.8940,58.6063,best_validation_chrF++



Saved: /kaggle/working/experiment1_indictrans2_lora/FINAL_full_validation_results.csv


In [14]:
# =============================================================================
# CELL 12 — FINAL TEST EVALUATION
#
# ONLY the best validation-chrF++ checkpoint is evaluated.
#
# Test set is completely untouched until this cell.
#
# Metrics:
#   BLEU
#   chrF++
#
# Beam size = 5.
# =============================================================================

import os
import gc
import pandas as pd
import torch

from transformers import (
    AutoModelForSeq2SeqLM
)

from peft import (
    PeftModel
)


FINAL_TEST_RESULTS = []


for direction in DIRECTIONS:

    print("\n")
    print("#" * 100)

    print(
        f"FINAL TEST: {direction}"
    )

    print("#" * 100)


    # =========================================================================
    # BEST ADAPTER
    # =========================================================================

    best_adapter_dir = os.path.join(
        DIRECTION_OUTPUT_ROOT,
        direction,
        "best_adapter",
    )


    if not os.path.exists(
        best_adapter_dir
    ):

        raise FileNotFoundError(
            f"Missing best adapter:\n"
            f"{best_adapter_dir}"
        )


    # =========================================================================
    # FRESH BASE
    # =========================================================================

    print(
        "\nLoading fresh base model..."
    )


    base_model = (
        AutoModelForSeq2SeqLM.from_pretrained(

            MODEL_NAME,

            trust_remote_code=True,

            torch_dtype=MODEL_DTYPE,

            token=HF_TOKEN,

        )
        .to(DEVICE)
    )


    # =========================================================================
    # LOAD LoRA
    # =========================================================================

    model = PeftModel.from_pretrained(
        base_model,
        best_adapter_dir,
    )


    model.eval()


    print(
        "Loaded adapter:",
        best_adapter_dir
    )


    # =========================================================================
    # TEST
    # =========================================================================

    info = direction_data[
        direction
    ]


    result = evaluate_direction(

        model=model,

        df=info["test"],

        src_lang=info["src_lang"],

        tgt_lang=info["tgt_lang"],

        direction_name=direction,

        max_examples=None,
    )


    # =========================================================================
    # STORE
    # =========================================================================

    FINAL_TEST_RESULTS.append({

        "direction":
            direction,

        "N":
            result["n"],

        "BLEU":
            result["BLEU"],

        "chrF++":
            result["chrF++"],

        "checkpoint":
            "best_validation_chrF++",

    })


    print("\nRESULTS")

    print(
        f"BLEU   : "
        f"{result['BLEU']:.2f}"
    )

    print(
        f"chrF++ : "
        f"{result['chrF++']:.2f}"
    )


    # =========================================================================
    # CLEANUP
    # =========================================================================

    del model
    del base_model

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# =============================================================================
# SAVE
# =============================================================================

final_test_df = pd.DataFrame(
    FINAL_TEST_RESULTS
)


final_test_path = os.path.join(
    OUTPUT_DIR,
    "FINAL_directional_lora_test_results.csv"
)


final_test_df.to_csv(
    final_test_path,
    index=False
)


print("\n")
print("=" * 100)
print("FINAL LoRA TEST RESULTS")
print("=" * 100)


display(
    final_test_df.round(4)
)


print(
    "\nSaved:",
    final_test_path
)



####################################################################################################
FINAL TEST: guj2mar
####################################################################################################

Loading fresh base model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded adapter: /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/guj2mar/best_adapter


Translating guj → mar:   0%|          | 0/84 [00:00<?, ?it/s]


RESULTS
BLEU   : 47.94
chrF++ : 63.88


####################################################################################################
FINAL TEST: mar2guj
####################################################################################################

Loading fresh base model...
Loaded adapter: /kaggle/working/experiment1_indictrans2_lora/direction_specific_lora_320m_v2_translit/mar2guj/best_adapter


Translating mar → guj:   0%|          | 0/84 [00:00<?, ?it/s]

... (3 outputs left out)

Translating pan → doi:   0%|          | 0/73 [00:00<?, ?it/s]


RESULTS
BLEU   : 44.66
chrF++ : 57.64


FINAL LoRA TEST RESULTS


,direction,N,BLEU,chrF++,checkpoint
0,guj2mar,1329,47.9356,63.8829,best_validation_chrF++
1,mar2guj,1329,51.2262,64.7281,best_validation_chrF++
2,doi2pan,1158,51.3911,64.2072,best_validation_chrF++
3,pan2doi,1158,44.6596,57.6376,best_validation_chrF++



Saved: /kaggle/working/experiment1_indictrans2_lora/FINAL_directional_lora_test_results.csv


In [15]:
# =============================================================================
# CELL 13 — FINAL BASE vs LoRA COMPARISON
#
# Base IndicTrans2
#        VS
# Best Direction-specific LoRA
#
# Metrics:
#   BLEU
#   chrF++
# =============================================================================

import os
import gc
import pandas as pd
import torch

from transformers import (
    AutoModelForSeq2SeqLM
)


# =============================================================================
# 1. BASE TEST EVALUATION
# =============================================================================

print("=" * 100)
print(
    "EVALUATING BASE IndicTrans2 "
    "ON HELD-OUT TEST"
)
print("=" * 100)


base_model = (
    AutoModelForSeq2SeqLM.from_pretrained(

        MODEL_NAME,

        trust_remote_code=True,

        torch_dtype=MODEL_DTYPE,

        token=HF_TOKEN,

    )
    .to(DEVICE)
)


base_model.eval()


BASE_TEST_RESULTS = []


for direction in DIRECTIONS:

    info = direction_data[
        direction
    ]


    result = evaluate_direction(

        model=base_model,

        df=info["test"],

        src_lang=info["src_lang"],

        tgt_lang=info["tgt_lang"],

        direction_name=direction,

        max_examples=None,
    )


    BASE_TEST_RESULTS.append({

        "model":
            "Base IndicTrans2",

        "direction":
            direction,

        "N":
            result["n"],

        "BLEU":
            result["BLEU"],

        "chrF++":
            result["chrF++"],

    })


    print(
        f"\n{direction}"
    )

    print(
        f"BLEU   : "
        f"{result['BLEU']:.2f}"
    )

    print(
        f"chrF++ : "
        f"{result['chrF++']:.2f}"
    )


del base_model

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# =============================================================================
# 2. LoRA TEST RESULTS
# =============================================================================

lora_test_df = pd.DataFrame(
    FINAL_TEST_RESULTS
)


lora_test_df = lora_test_df[
    [
        "direction",
        "N",
        "BLEU",
        "chrF++",
    ]
].copy()


lora_test_df[
    "model"
] = "Best Direction-specific LoRA"


# =============================================================================
# 3. BASE DATAFRAME
# =============================================================================

base_test_df = pd.DataFrame(
    BASE_TEST_RESULTS
)


# =============================================================================
# 4. COMBINE
# =============================================================================

comparison_df = pd.concat(

    [
        base_test_df,

        lora_test_df[
            [
                "model",
                "direction",
                "N",
                "BLEU",
                "chrF++",
            ]
        ],
    ],

    ignore_index=True,
)


# =============================================================================
# 5. CALCULATE IMPROVEMENTS
# =============================================================================

base_lookup = (
    base_test_df
    .set_index("direction")
)


lora_lookup = (
    lora_test_df
    .set_index("direction")
)


improvement_rows = []


for direction in DIRECTIONS:

    improvement_rows.append({

        "direction":
            direction,

        "Base BLEU":
            base_lookup.loc[
                direction,
                "BLEU"
            ],

        "LoRA BLEU":
            lora_lookup.loc[
                direction,
                "BLEU"
            ],

        "Δ BLEU":
            (
                lora_lookup.loc[
                    direction,
                    "BLEU"
                ]
                -
                base_lookup.loc[
                    direction,
                    "BLEU"
                ]
            ),

        "Base chrF++":
            base_lookup.loc[
                direction,
                "chrF++"
            ],

        "LoRA chrF++":
            lora_lookup.loc[
                direction,
                "chrF++"
            ],

        "Δ chrF++":
            (
                lora_lookup.loc[
                    direction,
                    "chrF++"
                ]
                -
                base_lookup.loc[
                    direction,
                    "chrF++"
                ]
            ),
    })


improvement_df = pd.DataFrame(
    improvement_rows
)


# =============================================================================
# 6. SAVE
# =============================================================================

comparison_path = os.path.join(
    OUTPUT_DIR,
    "FINAL_base_vs_directional_lora.csv"
)


improvement_path = os.path.join(
    OUTPUT_DIR,
    "FINAL_directional_lora_improvements.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


improvement_df.to_csv(
    improvement_path,
    index=False
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print("\n")
print("=" * 100)
print("BASE vs BEST DIRECTION-SPECIFIC LoRA")
print("=" * 100)


display(
    comparison_df.round(4)
)


print("\n")
print("=" * 100)
print("IMPROVEMENT OVER BASE")
print("=" * 100)


display(
    improvement_df.round(4)
)


print("\n")
print("=" * 100)
print("FINAL FILES")
print("=" * 100)


print(
    comparison_path
)

print(
    improvement_path
)

EVALUATING BASE IndicTrans2 ON HELD-OUT TEST


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Translating guj → mar:   0%|          | 0/84 [00:00<?, ?it/s]


guj2mar
BLEU   : 41.40
chrF++ : 58.61


Translating mar → guj:   0%|          | 0/84 [00:00<?, ?it/s]


mar2guj
BLEU   : 45.17
chrF++ : 60.20


... (4 outputs left out)

,model,direction,N,BLEU,chrF++
0,Base IndicTrans2,guj2mar,1329,41.4036,58.6051
1,Base IndicTrans2,mar2guj,1329,45.1719,60.2048
2,Base IndicTrans2,doi2pan,1158,44.6472,58.5790
3,Base IndicTrans2,pan2doi,1158,38.9106,52.8494
4,Best Direction-specific LoRA,guj2mar,1329,47.9356,63.8829
5,Best Direction-specific LoRA,mar2guj,1329,51.2262,64.7281
6,Best Direction-specific LoRA,doi2pan,1158,51.3911,64.2072
7,Best Direction-specific LoRA,pan2doi,1158,44.6596,57.6376




IMPROVEMENT OVER BASE


,direction,Base BLEU,LoRA BLEU,Δ BLEU,Base chrF++,LoRA chrF++,Δ chrF++
0,guj2mar,41.4036,47.9356,6.5320,58.6051,63.8829,5.2777
1,mar2guj,45.1719,51.2262,6.0543,60.2048,64.7281,4.5233
2,doi2pan,44.6472,51.3911,6.7439,58.5790,64.2072,5.6281
3,pan2doi,38.9106,44.6596,5.7489,52.8494,57.6376,4.7882




FINAL FILES
/kaggle/working/experiment1_indictrans2_lora/FINAL_base_vs_directional_lora.csv
/kaggle/working/experiment1_indictrans2_lora/FINAL_directional_lora_improvements.csv


In [16]:
# =============================================================================
# CELL 14 — FINAL EXPERIMENT SUMMARY
#
# Metrics:
#   BLEU
#   chrF++
# =============================================================================

print("=" * 100)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 100)


summary = improvement_df.copy()


summary[
    "BLEU improved?"
] = (
    summary["Δ BLEU"] > 0
)


summary[
    "chrF++ improved?"
] = (
    summary["Δ chrF++"] > 0
)


display(
    summary.round(4)
)


print(
    "\nDirection-level improvements:"
)


for _, row in summary.iterrows():

    print(
        f"\n{row['direction']}"
    )

    print(
        f"  Δ BLEU   : "
        f"{row['Δ BLEU']:+.2f}"
    )

    print(
        f"  Δ chrF++ : "
        f"{row['Δ chrF++']:+.2f}"
    )

FINAL EXPERIMENT SUMMARY


,direction,Base BLEU,LoRA BLEU,Δ BLEU,Base chrF++,LoRA chrF++,Δ chrF++,BLEU improved?,chrF++ improved?
0,guj2mar,41.4036,47.9356,6.5320,58.6051,63.8829,5.2777,True,True
1,mar2guj,45.1719,51.2262,6.0543,60.2048,64.7281,4.5233,True,True
2,doi2pan,44.6472,51.3911,6.7439,58.5790,64.2072,5.6281,True,True
3,pan2doi,38.9106,44.6596,5.7489,52.8494,57.6376,4.7882,True,True



Direction-level improvements:

guj2mar
  Δ BLEU   : +6.53
  Δ chrF++ : +5.28

mar2guj
  Δ BLEU   : +6.05
  Δ chrF++ : +4.52

doi2pan
  Δ BLEU   : +6.74
  Δ chrF++ : +5.63

pan2doi
  Δ BLEU   : +5.75
  Δ chrF++ : +4.79
